# Own Analysis Week 2: Ribosome footprints distribution

Based on the Week 2 Guided Mission (calculating RPF distribution around the start codon) and our Own Analysis from Week 1 (classifying genes into Lin28a targets and non-targets), we will investigate whether the Ribosome protected fragment (RPF) distribution differs between Lin28a targets and non-targets.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.rcParams['figure.figsize'] = [10, 6]
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

## 1. Classify Genes as Lin28a Targets vs Non-targets
We use the CLIP-seq data from `read-counts.txt` to identify direct targets.

In [ ]:
# Load read counts, skip the first row (featureCounts command comment)
df = pd.read_csv('binfo1-datapack1/read-counts.txt', sep='\t', comment='#')
df = df.rename(columns={'CLIP-35L33G.bam': 'CLIP'})

def label_target(clip_count):
    if clip_count >= 30:
        return 'Target'
    elif clip_count == 0:
        return 'Non-target'
    else:
        return 'Intermediate'

df['Lin28a_Binding'] = df['CLIP'].apply(label_target)

target_genes = set(df[df['Lin28a_Binding'] == 'Target']['Geneid'])
nontarget_genes = set(df[df['Lin28a_Binding'] == 'Non-target']['Geneid'])
print(f"Found {len(target_genes)} targets and {len(nontarget_genes)} non-targets.")

## 2. Map Transcript IDs to Gene IDs
The Week 2 alignment produced `fivepcounts-filtered-RPF-siLuc.txt` which contains `transcript_id`s. We need to map these to `gene_id`s to cross-reference with our CLIP data.

In [ ]:
transcript_to_gene = {}
with open('binfo1-datapack1/gencode.gtf', 'r') as f:
    for line in f:
        if line.startswith('#'):
            continue
        parts = line.split('\t')
        if len(parts) > 8 and parts[2] == 'transcript':
            attrs = parts[8]
            gene_id = [x.split('"')[1] for x in attrs.split(';') if 'gene_id' in x][0]
            transcript_id = [x.split('"')[1] for x in attrs.split(';') if 'transcript_id' in x][0]
            transcript_to_gene[transcript_id] = gene_id
            
print(f"Loaded {len(transcript_to_gene)} transcript mappings.")

## 3. Extract RPF Relative Positions
Read the counts and group by Target vs Non-target.

In [ ]:
from collections import Counter

target_counts = Counter()
nontarget_counts = Counter()

with open('binfo1-work/fivepcounts-filtered-RPF-siLuc.txt', 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) < 10:
            continue
            
        pos = int(parts[1])
        count = int(parts[3])
        transcript_id = parts[7]
        start_codon_start = int(parts[8])
        
        rel_pos = pos - start_codon_start
        
        gene_id = transcript_to_gene.get(transcript_id)
        if gene_id in target_genes:
            target_counts[rel_pos] += count
        elif gene_id in nontarget_genes:
            nontarget_counts[rel_pos] += count

print("Processed RPF counts.")

## 4. Plotting the RPF Distribution
We normalize the read counts to compare the footprint distributions and their 3-nt periodicity between targets and non-targets.

In [ ]:
positions = list(range(-50, 50))
tc = [target_counts[p] for p in positions]
ntc = [nontarget_counts[p] for p in positions]

# Normalize by the sum so we can compare distributions
tc_sum = sum(tc) if sum(tc) > 0 else 1
ntc_sum = sum(ntc) if sum(ntc) > 0 else 1

tc_norm = [c / tc_sum for c in tc]
ntc_norm = [c / ntc_sum for c in ntc]

plt.figure(figsize=(12, 6))
plt.plot(positions, tc_norm, label='Lin28a Targets', color='red', alpha=0.8, linewidth=2)
plt.plot(positions, ntc_norm, label='Non-targets', color='gray', alpha=0.8, linewidth=2)
plt.xlim(-50, 50)
plt.xlabel("Relative position to start codon of 5'-ends of reads")
plt.ylabel('Normalized read fraction')
plt.title('RPF-siLuc Distribution around Start Codon (Targets vs Non-targets)')
plt.legend()
plt.axvline(0, color='black', linestyle='--')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()